# EduRecSys – Data Preprocessing & Unified Schema

This notebook performs data preprocessing and feature engineering for the EduRecSys project.

The main objectives of this notebook are to:
- Transform the Udemy (English) and SANAD (Arabic) datasets into a unified data schema.
- Generate consistent features required for recommendation modeling.
- Handle multilingual data by adding language-aware attributes.
- Prepare clean, structured datasets for downstream recommendation algorithms.

At the end of this notebook, two processed datasets will be produced:
- `content_en.csv` for English educational content.
- `content_ar.csv` for Arabic educational content.

Unified Schema:

- content_id
- title
- description
- subject
- level
- content_duration
- language

In [1]:
import os
import pandas as pd
from pathlib import Path
import numpy as np

BASE_PATH = Path("/kaggle/input/edurecsys-raw-data")
OUTPUT_PATH = Path("/kaggle/working")

UDEMY_PATH = BASE_PATH / "udemy_courses.csv"
SANAD_PATH = BASE_PATH / "/kaggle/input/edurecsys-raw-data/sanad/sanad"


In [2]:
udemy_df = pd.read_csv(UDEMY_PATH)
sanad_records = []

for subject in ["Tech", "Finance"]:
    subject_path = SANAD_PATH / subject
    for file_name in os.listdir(subject_path):
        if file_name.endswith(".txt"):
            with open(subject_path / file_name, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read().strip()
            sanad_records.append({
                "content_id": f"{subject}_{file_name.replace('.txt','')}",
                "description": text,
                "subject": subject
            })

sanad_df = pd.DataFrame(sanad_records)

print(udemy_df.shape, sanad_df.shape)


(3678, 12) (13000, 3)


In [6]:
# Udemy → Unified Schema
udemy_unified = pd.DataFrame({
    "content_id": udemy_df["course_id"],
    "title": udemy_df["course_title"],
    "description": udemy_df["course_title"],  # Udemy has no long description
    "subject": udemy_df["subject"],
    "level": udemy_df["level"],
    "content_duration": udemy_df["content_duration"],
    "language": "en"
})


In [4]:
# SANAD → Unified Schema
sanad_unified = sanad_df.copy()

sanad_unified["title"] = sanad_unified["description"].apply(lambda x: " ".join(x.split()[:8]))
sanad_unified["level"] = sanad_unified["description"].apply(
    lambda x: "Beginner" if len(x.split()) < 300 else "Intermediate"
)
sanad_unified["content_duration"] = sanad_unified["description"].apply(
    lambda x: round(len(x.split()) / 200, 2)
)
sanad_unified["language"] = "ar"

sanad_unified = sanad_unified[[
    "content_id", "title", "description",
    "subject", "level", "content_duration", "language"
]]


In [8]:
udemy_unified.head()

,content_id,title,description,subject,level,content_duration,language
0,1070968,Ultimate Investment Banking Course,Ultimate Investment Banking Course,Business Finance,All Levels,1.5,en
1,1113822,Complete GST Course & Certification - Grow You...,Complete GST Course & Certification - Grow You...,Business Finance,All Levels,39.0,en
2,1006314,Financial Modeling for Business Analysts and C...,Financial Modeling for Business Analysts and C...,Business Finance,Intermediate Level,2.5,en
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,Beginner to Pro - Financial Analysis in Excel ...,Business Finance,All Levels,3.0,en
4,1011058,How To Maximize Your Profits Trading Options,How To Maximize Your Profits Trading Options,Business Finance,Intermediate Level,2.0,en


In [9]:
sanad_unified.head()

,content_id,title,description,subject,level,content_duration,language
0,Tech_1893,تستعد اتصالات لطرح نظام التحاسب بالثانية لجميع...,تستعد اتصالات لطرح نظام التحاسب بالثانية لجميع...,Tech,Intermediate,2.07,ar
1,Tech_1711,أعلنت وحدة الأعمال المتنقلة التابعة لشركة موتو...,أعلنت وحدة الأعمال المتنقلة التابعة لشركة موتو...,Tech,Intermediate,1.72,ar
2,Tech_4682,تطالب شركة جوجل، صاحبة محرك البحث الشهير على,تطالب شركة جوجل، صاحبة محرك البحث الشهير على ا...,Tech,Beginner,0.91,ar
3,Tech_5450,اختارت شركة الثريا للاتصالات شركة فورت انفو تك...,اختارت شركة الثريا للاتصالات شركة فورت انفو تك...,Tech,Beginner,0.97,ar
4,Tech_5064,أعلنت دو أنها ستبدأ قريباً بدعوة عُملائها لتسجيل,أعلنت دو أنها ستبدأ قريباً بدعوة عُملائها لتسج...,Tech,Beginner,0.71,ar


In [7]:
# Save Outputs
udemy_unified.to_csv(OUTPUT_PATH / "content_en.csv", index=False)
sanad_unified.to_csv(OUTPUT_PATH / "content_ar.csv", index=False)
